IS I170 LAB 9

SWATI, ALEXIS VELASQUEZ

MutinominalNB https://archive.ics.uci.edu/dataset/228/sms+spam+collection

 SMS Spam Classification using Multinomial Naive Bayes

In [ ]:
# Import libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
# Load dataset

#Upload file in the colab first

from google.colab import files
uploaded = files.upload()

Saving SMSSpamCollection to SMSSpamCollection


In [ ]:
# Read file

df = pd.read_csv("SMSSpamCollection", sep='\t', header=None)

In [ ]:
# Add column names

df.columns = ['label', 'message']


In [ ]:
#Convert labels

df['label']= df.label.map({'ham':0, 'spam':1})

df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
#  Clean text

import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()                      # make all text lowercase
    text = re.sub(r'\W', ' ', text)          # remove punctuation
    text = re.sub(r'\d+', '', text)          # remove numbers
    words = text.split()                     # split sentence into words
    words = [w for w in words if w not in stop_words]  # remove common words
    return " ".join(words)

# create a new cleaned text column
df['cleaned_message'] = df['message'].apply(clean_text)

# check that it worked
print(df[['message', 'cleaned_message']].head())

                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                     cleaned_message  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry wkly comp win fa cup final tkts st ...  
3                u dun say early hor u c already say  
4             nah think goes usf lives around though  


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
#Split Data
#X = input (text)
#y = output (spam or not)

X = df['cleaned_message']
y= df['label']

#Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#Convert text into numbers

#Machine Learning models cannot read text directly
#TF-IDF converts words into numeric importance values

vectorizer = CountVectorizer()

#Learn from training data and transform both sets
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
#Train MultinominalNB model
#This model works well with word frequencies

model = MultinomialNB()

#Train model using training data
model.fit(X_train_vec, y_train)

MultinomialNB()

In [ ]:
#Make predictions

#Predict Labels for test data

y_pred = model.predict(X_test_vec)


In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Convert to table
cm_df = pd.DataFrame(
    cm,
    index=["Actual: Not Spam", "Actual: Spam"],
    columns=["Predicted: Not Spam", "Predicted: Spam"]
)

# Display normal table
print("Confusion Matrix:")
display(cm_df)

# Display colored version
display(
    cm_df.style
    .background_gradient(cmap='Blues')
    .set_caption("📊 Confusion Matrix")
)

Confusion Matrix:


,Predicted: Not Spam,Predicted: Spam
Actual: Not Spam,957,9
Actual: Spam,10,139


,Predicted: Not Spam,Predicted: Spam
Actual: Not Spam,957,9
Actual: Spam,10,139


In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# Convert report to dictionary, then DataFrame
report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()

# Show clean table
print("Classification Report:")
display(report_df)

display(
    report_df.style
    .background_gradient(cmap='viridis')   # color gradient
    .set_caption("📊 Classification Report")
    .format("{:.3f}")  # round numbers nicely
)

Classification Report:


,precision,recall,f1-score,support
0,0.989659,0.990683,0.990171,966.00000
1,0.939189,0.932886,0.936027,149.00000
accuracy,0.982960,0.982960,0.982960,0.98296
macro avg,0.964424,0.961785,0.963099,1115.00000
weighted avg,0.982914,0.982960,0.982935,1115.00000


,precision,recall,f1-score,support
0,0.990,0.991,0.990,966.000
1,0.939,0.933,0.936,149.000
accuracy,0.983,0.983,0.983,0.983
macro avg,0.964,0.962,0.963,1115.000
weighted avg,0.983,0.983,0.983,1115.000


In [ ]:
# ---------------------------
# STEP 10: Test new message
# ---------------------------

# New message (MUST be inside a list and quotes)
new_message = ["Congratulations! You have won a free prize!"]

# Clean the message using the SAME function
cleaned = clean_text(new_message[0])

# Convert text into numerical format
new_vec = vectorizer.transform([cleaned])

# Predict
prediction = model.predict(new_vec)

# Show result
print("\nNew Message Prediction:")
if prediction[0] == 1:
    print("Spam message")
else:
    print("Not Spam")


New Message Prediction:
Spam message


In [ ]:
# STEP 12: Test more than one new message
new_messages = [
    "Free entry in 2 a wkly comp to win FA Cup tickets",
    "Hey, are we still meeting after class today?",
    "Claim your prize now by clicking this link"
]

# Clean each message
cleaned_messages = [clean_text(msg) for msg in new_messages]

# Convert to numbers
new_messages_vec = vectorizer.transform(cleaned_messages)

# Predict
predictions = model.predict(new_messages_vec)

# Show results in a table
results_df = pd.DataFrame({
    "Original Message": new_messages,
    "Prediction": ["Spam" if p == 1 else "Not Spam" for p in predictions]
})

display(results_df)

,Original Message,Prediction
0,Free entry in 2 a wkly comp to win FA Cup tickets,Spam
1,"Hey, are we still meeting after class today?",Not Spam
2,Claim your prize now by clicking this link,Spam


In [ ]:
# STEP 13: Make the results table prettier
display(
    results_df.style
    .background_gradient(cmap="Greens")
    .set_caption("New Message Predictions")
)

,Original Message,Prediction
0,Free entry in 2 a wkly comp to win FA Cup tickets,Spam
1,"Hey, are we still meeting after class today?",Not Spam
2,Claim your prize now by clicking this link,Spam


In this lab, a Multinomial Naive Bayes model for text categorization was created using the SMS Spam dataset. The dataset was appropriate for classification tasks because it included labeled messages classified as spam or non spam. To enhance model performance, a number of data engineering techniques were used, such as changing all text to lowercase, deleting common stopwords, and removing punctuation and digits. By doing these actions, the data's noise was reduced and the emphasis was placed on terms that were significant.
TF-IDF vectorization, which captures the significance of words based on their frequency across messages, was then used to convert the cleaned text into numerical characteristics. The Multinomial Naive Bayes algorithm was used because it works well with textual and count-based data. Accuracy, confusion matrix, and classification report were used to assess the model after it was trained on the processed dataset. According to the findings, the model performed well in differentiating between spam and non-spam messages, properly classifying the majority of messages. The trained model was also used to test additional message inputs to mimic real-world usage, demonstrating the model's ability to generalize to new data.

Overall, this experiment demonstrates how Multinomial Naive Bayes, combined with proper data preprocessing and feature extraction, can effectively solve text classification problems in practical applications such as spam detection.